In [1]:
#!pip install pandas pymongo fastparquet
!pip install pandas pymongo sqlalchemy pyodbc fastparquet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from pymongo import MongoClient
import pymongo 
import urllib 
import sqlalchemy 
import time

# Start the timer
start_time = time.time()

# --- Configuration ---
SQL_SERVER_HOST = "HYPE\\SQL_2022"
SQL_SERVER_DB = "Sttwk_2022a"
SQL_SERVER_USER = "sa"
SQL_SERVER_PASSWORD = "D@1ry$2014"
SQL_TABLE_NAME = "sttwk"
#SQL_PRIMARY_KEY_COL = "prod_id" 
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "sttwk" 

quoted_password = urllib.parse.quote_plus(SQL_SERVER_PASSWORD)
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_SERVER_HOST};DATABASE={SQL_SERVER_DB};UID={SQL_SERVER_USER};PWD={quoted_password};ENCRYPT=yes;TrustServerCertificate=yes'
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")

try:
    df = pd.read_sql_table(SQL_TABLE_NAME, con=engine)
    print(f"Successfully loaded {len(df)} rows from SQL Server table '{SQL_TABLE_NAME}'.")
except Exception as e:
    print(f"Error reading SQL table: {e}")
    exit()

# --- 2. Clean and Prepare Data for MongoDB ---

# --- FIX: Convert Pandas NaT (Null Dates) to None (MongoDB Null) using a compatible method ---
for col in df.columns:
    # Check if the dtype string contains "datetime"
    if "datetime" in str(df[col].dtype):
        df[col] = df[col].replace({pd.NaT: None})

# MongoDB _id cannot be null/empty. Filter out any bad primary keys.
#df_cleaned = df.dropna(subset=[SQL_PRIMARY_KEY_COL])
#df_cleaned = df_cleaned[df_cleaned[SQL_PRIMARY_KEY_COL] != '']

#print(f"Number of valid rows after cleaning primary key column: {len(df_cleaned)}")

# Convert DataFrame to a list of dictionaries (MongoDB documents)
#data_to_insert = df_cleaned.to_dict(orient='records')
df_to_insert = df.copy()
# Convert Pandas NaT (Null Dates) to None (MongoDB Null) for compatibility
for col in df_to_insert.columns:
    if "datetime" in str(df_to_insert[col].dtype):
        df_to_insert[col] = df_to_insert[col].replace({pd.NaT: None})
# Convert DataFrame to a list of dictionaries (MongoDB documents)
data_to_insert = df_to_insert.to_dict(orient='records')

print(f"Number of documents ready for insertion: {len(data_to_insert)}")        

# Copy the primary key value into the MongoDB '_id' field
#for doc in data_to_insert:
#    doc['_id'] = doc[SQL_PRIMARY_KEY_COL]

# --- 3. Connect to MongoDB and Insert Data ---

try:
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    
    collection.delete_many({}) 
    
    #collection.insert_many(data_to_insert)
    collection.insert_many(df_to_insert)
    print(f"Successfully inserted {len(df_to_insert)} documents into MongoDB collection '{MONGO_COLLECTION}'.")
    
    # Create a secondary unique index on the original column name
    #print(f"Creating secondary unique index on '{SQL_PRIMARY_KEY_COL}' field...")
    #collection.create_index(
    #    [(SQL_PRIMARY_KEY_COL, pymongo.ASCENDING)], 
    #    unique=True
    #)
    #print("Index created successfully.")
    
except Exception as e:
    # Print the specific error that halted the insertion
    print(f"An error occurred during MongoDB insertion or indexing: {e}")
finally:
    if client:
        client.close()

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")


Successfully loaded 7142204 rows from SQL Server table 'sttwk'.
Number of documents ready for insertion: 7142204
An error occurred during MongoDB insertion or indexing: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Total time taken for transfer: 349.62 seconds


In [4]:
import pandas as pd
from pymongo import MongoClient
import pymongo 
import urllib 
import sqlalchemy 
import time

# Start the timer
start_time = time.time()

# --- Configuration ---
SQL_SERVER_HOST = "HYPE\\SQL_2022"
SQL_SERVER_DB = "Sttwk_2022a"
SQL_SERVER_USER = "sa"
SQL_SERVER_PASSWORD = "D@1ry$2014"
SQL_TABLE_NAME = "sttwk"
#SQL_PRIMARY_KEY_COL = "prod_id" 
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "sttwk" 

quoted_password = urllib.parse.quote_plus(SQL_SERVER_PASSWORD)
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_SERVER_HOST};DATABASE={SQL_SERVER_DB};UID={SQL_SERVER_USER};PWD={quoted_password};ENCRYPT=yes;TrustServerCertificate=yes'
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")

try:
    df = pd.read_sql_table(SQL_TABLE_NAME, con=engine)
    print(f"Successfully loaded {len(df)} rows from SQL Server table '{SQL_TABLE_NAME}'.")
except Exception as e:
    print(f"Error reading SQL table: {e}")
    exit()

# --- 2. Clean and Prepare Data for MongoDB ---

# --- FIX: Convert Pandas NaT (Null Dates) to None (MongoDB Null) using a compatible method ---
for col in df.columns:
    # Check if the dtype string contains "datetime"
    if "datetime" in str(df[col].dtype):
        df[col] = df[col].replace({pd.NaT: None})

# MongoDB _id cannot be null/empty. Filter out any bad primary keys.
#df_cleaned = df.dropna(subset=[SQL_PRIMARY_KEY_COL])
#df_cleaned = df_cleaned[df_cleaned[SQL_PRIMARY_KEY_COL] != '']

#print(f"Number of valid rows after cleaning primary key column: {len(df_cleaned)}")

# Convert DataFrame to a list of dictionaries (MongoDB documents)
#data_to_insert = df_cleaned.to_dict(orient='records')
df_to_insert = df.copy()
# Convert Pandas NaT (Null Dates) to None (MongoDB Null) for compatibility
for col in df_to_insert.columns:
    if "datetime" in str(df_to_insert[col].dtype):
        df_to_insert[col] = df_to_insert[col].replace({pd.NaT: None})
# Convert DataFrame to a list of dictionaries (MongoDB documents)
data_to_insert = df_to_insert.to_dict(orient='records')

print(f"Number of documents ready for insertion: {len(data_to_insert)}")        

# Copy the primary key value into the MongoDB '_id' field
#for doc in data_to_insert:
#    doc['_id'] = doc[SQL_PRIMARY_KEY_COL]

# --- 3. Connect to MongoDB and Insert Data ---

try:
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    
    collection.delete_many({}) 
    
    print(f"Starting bulk insert of {len(data_to_insert)} documents...")
    collection.insert_many(data_to_insert)
    # This insertion might take several minutes
    print(f"Successfully inserted {len(data_to_insert)} documents into MongoDB collection '{MONGO_COLLECTION}'.")
    
    # Create a secondary unique index on the original column name
    #print(f"Creating secondary unique index on '{SQL_PRIMARY_KEY_COL}' field...")
    #collection.create_index(
    #    [(SQL_PRIMARY_KEY_COL, pymongo.ASCENDING)], 
    #    unique=True
    #)
    #print("Index created successfully.")
    
except Exception as e:
    # Print the specific error that halted the insertion
    print(f"An error occurred during MongoDB insertion or indexing: {e}")
finally:
    if client:
        client.close()

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")


Successfully loaded 7142204 rows from SQL Server table 'sttwk'.
Number of documents ready for insertion: 7142204
Starting bulk insert of 7142204 documents...
Successfully inserted 7142204 documents into MongoDB collection 'sttwk'.
Total time taken for transfer: 1180.25 seconds


In [5]:
from pymongo import MongoClient
import time

# Start the timer
start_time = time.time()

MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "sttwk"

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[MONGO_COLLECTION]

pipeline = [
    {
        '$group': {
            '_id': {
                'year': '$year',
                'week': '$week'
            },
            'totalNetQty': {
                '$sum': '$net_qty'
            }
        }
    },
    {
        '$sort': {
            '_id.year': 1,
            '_id.week': 1
        }
    },
    {
        '$project': {
            '_id': 0,
            'year': '$_id.year',
            'week': '$_id.week',
            'sum_net_qty': '$totalNetQty'
        }
    }
]

# Execute the aggregation query
results = collection.aggregate(pipeline)

# Print the results
for doc in results:
    print(doc)

client.close()

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for query: {total_time:.2f} seconds")

{'year': '2022', 'week': '01', 'sum_net_qty': 1176184.497}
{'year': '2022', 'week': '02', 'sum_net_qty': 1450239.337}
{'year': '2022', 'week': '03', 'sum_net_qty': 1337833.07}
{'year': '2022', 'week': '04', 'sum_net_qty': 1625286.49}
{'year': '2022', 'week': '05', 'sum_net_qty': 878449.046}
{'year': '2022', 'week': '06', 'sum_net_qty': 1382949.051}
{'year': '2022', 'week': '07', 'sum_net_qty': 1459358.26}
{'year': '2022', 'week': '08', 'sum_net_qty': 1767098.074}
{'year': '2022', 'week': '09', 'sum_net_qty': 840785.073}
{'year': '2022', 'week': '10', 'sum_net_qty': 1462618.241}
{'year': '2022', 'week': '11', 'sum_net_qty': 1452046.713}
{'year': '2022', 'week': '12', 'sum_net_qty': 1754371.831}
{'year': '2022', 'week': '13', 'sum_net_qty': 2204147.073}
Total time taken for query: 5.77 seconds


In [6]:
import pandas as pd
import urllib
import sqlalchemy
import time

# Start the timer
start_time = time.time()

# --- Configuration ---
SQL_SERVER_HOST = "HYPE\\SQL_2022"
SQL_SERVER_DB = "Sttwk_2022a" # The source database
SQL_SERVER_USER = "sa"
SQL_SERVER_PASSWORD = "D@1ry$2014"

# --- 1. Define the SQL Query ---
# Note: SQL Server requires year/week to be extracted from a date field if they are not already columns
# I assume YEAR and WEEK are already columns in your table for this query to work as written:
sql_query = """
SELECT 
    [year], 
    [week], 
    SUM(net_qty) AS sum_net_qty
FROM 
    sttwk
GROUP BY 
    [year], 
    [week]
ORDER BY 
    [year], 
    [week];
"""

# --- 2. Connect to SQL Server and Execute the Query ---

quoted_password = urllib.parse.quote_plus(SQL_SERVER_PASSWORD)
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_SERVER_HOST};DATABASE={SQL_SERVER_DB};UID={SQL_SERVER_USER};PWD={quoted_password};ENCRYPT=yes;TrustServerCertificate=yes'
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")

try:
    # Use pandas to read the results of the SQL query directly into a DataFrame
    df_results = pd.read_sql_query(sql_query, con=engine)
    
    print(f"Query executed successfully. Results:")
    # Display all results (Pandas usually truncates output by default if many rows)
    print(df_results.to_string()) 

except Exception as e:
    print(f"Error executing SQL query: {e}")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for query: {total_time:.2f} seconds")

Query executed successfully. Results:
    year week  sum_net_qty
0   2022   01  1176184.497
1   2022   02  1450239.337
2   2022   03  1337833.070
3   2022   04  1625286.490
4   2022   05   878449.046
5   2022   06  1382949.051
6   2022   07  1459358.260
7   2022   08  1767098.074
8   2022   09   840785.073
9   2022   10  1462618.241
10  2022   11  1452046.713
11  2022   12  1754371.831
12  2022   13  2204147.073
Total time taken for query: 1.50 seconds
